## **Análisis de texto**

En el siguiente notebook se propone predecir los tiempos de adopción de las diferentes mascotas en base a la descripción disponible en su publicación. Para realizar esta tarea, se utilizarán modelos preentrenados y se realizarán algunas modificaciones y un poco de fine tuning de manera de adaptar dichos modelos a la problemática bajo estudio.

El notebook está preparado para evaluar 4 condiciones de experimentación:

* **Modelo DistilBERT base:** es el paradigma brindado por la cátedra el cual servirá como referencia al momento de realizar las comparaciones.
* **Modelo DistilBERT multilingual:** dado que existen descripciones en otros idiomas, se decidió estudiar si trabajar con un modelo más general que no analice solamente el inglés puede traer mejores resultados.
* **Modelo MiniLM-L12-v2:** modelo alternativo a distilBERT que también es capaz de analizar texto en múltiples idiomas.
* **Traducción + DistilBERT base:** como cuarto enfoque se propone utilizar las librerías langdetect y deeptranslator para traducir en todas las descripciones al inglés en forma automática y luego realizar la predicción con el enfoque inicial.

#### **Librerías a utilizar**


In [1]:
# Data processing
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from datasets import Dataset,  DatasetDict
#from UA_MDM_LDI_II.tutoriales.utils import plot_confusion_matrix

# Modeling
import torch
from torch.utils.data import DataLoader
from transformers import DistilBertTokenizerFast, DataCollatorWithPadding, AutoModelForSequenceClassification, get_scheduler, AutoTokenizer, DistilBertForSequenceClassification
from torch.optim import AdamW

# Progress bar
from tqdm.auto import tqdm

# Verificamos que CUDA está funcional
torch.cuda.is_available()

True

#### **Bajamos el modelo**

En esta sección se debe elegir cual de los 3 tokenizers se van a utilizar en función del modelo elegido

In [ ]:
from transformers import DistilBertTokenizerFast, AutoTokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
#tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-multilingual-cased")
#tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

#### **Directorios y condiciones de experimentación**

Además de definir los directorios de trabajo (Pensados para hacer la corrida en colab) se definen la semilla aleatoria, el tamaño del testing set y el tamaño del batch. Se plantea la posibilidad de utilizar 3 semillas diferentes las cuales nos serán útiles para tener una medida más confiable de cuál es el mejor enfoque.

In [3]:
# Paths
BASE_DIR = '/content/'
PATH_TO_TRAIN = os.path.join(BASE_DIR, "train.csv")
PATH_TO_TEMP_FILES = os.path.join(BASE_DIR, "work/optuna_temp_artifacts")
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, "work/optuna_artifacts")

# Parametros y variables
SEED = 20001101
#SEED = 19960824
#SEED = 19721106
TEST_SIZE = 0.2
BATCH_SIZE = 64

#### **Traducción**

Se importan las librerías necesarias y se define una función para realizar la traducción automática al ingles de las descripciones de las mascotas

In [4]:
# Código opcional para hacer la traducción
import pandas as pd
!pip install langdetect
from langdetect import detect
!pip install deep_translator
from deep_translator import GoogleTranslator


def translate_to_english(text):
    try:
        lang = detect(text)
        if lang == "en":
            return text  # si ya está en inglés, lo dejamos igual
        else:
            return GoogleTranslator(source="auto", target="en").translate(text)
    except Exception:
        return text  # en caso de error devolvemos el original


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 27.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=0de9972baa47be4940aea7fab6154c259d7678ca6ff2dcad047dcf2838bff65e
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 880.2 kB/s eta 0:00:00


#### **Carga y preparación de datos**

En esta sección se carga el conjunto de datos train el cual contiene las descripciones de las mascotas, se divide dicho conjunto en training y testing set y se los prepara para realizar el entrenamiento del modelo. Adicionalmente, hay 2 líneas de código comentadas la cual realizan la traducción al inglés en caso de que se quiera adoptar dicho enfoque.

In [ ]:
# Cargar los datos
train_df = pd.read_csv(PATH_TO_TRAIN)
train_df['labels'] = train_df["AdoptionSpeed"]

# Dividir los datos usando sklearn
train_df, test_df = train_test_split(train_df, test_size=TEST_SIZE, random_state=SEED, stratify=train_df.AdoptionSpeed)

# Eliminar registros con descrición nula
train_df = train_df[train_df['Description'].notnull()]
test_df = test_df[test_df['Description'].notnull()]

# Traducción al inglés
#train_df["Description"] = train_df["Description"].apply(translate_to_english).astype(str)
#test_df["Description"] = test_df["Description"].apply(translate_to_english).astype(str)

# Convertir a Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Combinar en un DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

# Codificar la columna de etiquetas como clases
dataset = dataset.class_encode_column('labels')

# Hacer una lista de columnas para remover antes de la tokenización
cols_to_remove = [col for col in dataset["train"].column_names if col != 'labels']
print(cols_to_remove)

Stringifying the column:   0%|          | 0/11982 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/11982 [00:00<?, ? examples/s]

Stringifying the column:   0%|          | 0/2998 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/2998 [00:00<?, ? examples/s]

['Type', 'Name', 'Age', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2', 'Color3', 'MaturitySize', 'FurLength', 'Vaccinated', 'Dewormed', 'Sterilized', 'Health', 'Quantity', 'Fee', 'State', 'RescuerID', 'VideoAmt', 'Description', 'PetID', 'PhotoAmt', 'AdoptionSpeed', '__index_level_0__']


In [6]:
# Obtener el objeto ClassLabel del conjunto de datos de entrenamiento
class_label = dataset["train"].features["labels"]

# Obtener las clases originales a partir del objeto ClassLabel
classes = class_label.names
classes

['0', '1', '2', '3', '4']

In [ ]:
# Tokenize and encode the dataset
def tokenize(batch):
    # Ensure tokenizer is loaded correctly in each process
    from transformers import DistilBertTokenizerFast
    tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
    #tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-multilingual-cased")
    #tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    tokenized_batch = tokenizer(batch["Description"], padding=True, truncation=True, max_length=512)
    return tokenized_batch

dataset_enc = dataset.map(tokenize, batched=True, remove_columns=cols_to_remove, num_proc=4)

# Set dataset format for PyTorch
dataset_enc.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

# Check the output
print(dataset_enc["train"].column_names)

Map (num_proc=4):   0%|          | 0/11982 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/2998 [00:00<?, ? examples/s]

['labels', 'input_ids', 'attention_mask']


In [8]:
# Instantiate a data collator with dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Create data loaders for to reshape data for PyTorch model
train_dataloader = DataLoader(
    dataset_enc["train"], shuffle=True, batch_size=BATCH_SIZE, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    dataset_enc["test"], batch_size=BATCH_SIZE, collate_fn=data_collator
)

In [ ]:
# Dynamically set number of class labels based on dataset
num_labels = dataset["train"].features['labels'].num_classes
print(f"Number of labels: {num_labels}")

# Load model
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased",
                                                           num_labels=num_labels)
#model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-multilingual-cased", num_labels=num_labels)
#model = AutoModelForSequenceClassification.from_pretrained("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", num_labels=num_labels)

Number of labels: 5


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


#### **Parámetros del modelo**

Se define trabajar para todos los experimentos con un learning_rate = 0.00005 y num_epochs = 20

In [10]:
# Model parameters
learning_rate = 5e-5
num_epochs = 20

# Create the optimizer
optimizer = AdamW(model.parameters(), lr=learning_rate)

# Further define learning rate scheduler
num_training_batches = len(train_dataloader)
num_training_steps = num_epochs * num_training_batches
lr_scheduler = get_scheduler(
    "linear",                   # linear decay
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)



In [11]:

# Set the device automatically (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Move model to device
model.to(device)


cuda


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


#### **Entrenamiento del modelo**

In [ ]:
progress_bar = tqdm(range(num_training_steps))

# Train the model with PyTorch training loop
model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

  0%|          | 0/3760 [00:00<?, ?it/s]

#### **Evaluación del modelo**

Se evalúa el modelo en base al quadratic weighted kappa alcanzado en el conjunto de testing

In [ ]:
from sklearn.metrics import cohen_kappa_score

# Inicializa listas para almacenar todas las predicciones y etiquetas
all_predictions = []
all_labels = []
all_probabilities = []

# Iteratively evaluate the model and collect predictions and labels
model.eval()
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    predictions = torch.argmax(logits, dim=-1)

    # Mover predicciones y etiquetas a CPU y convertir a numpy
    all_predictions.extend(predictions.cpu().numpy())
    all_labels.extend(batch["labels"].cpu().numpy())
    all_probabilities.extend(probabilities.cpu().numpy())

# Convertir listas a arrays de numpy
all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)
all_probabilities = np.array(all_probabilities)

# Calcular Quadratic Weighted Kappa
qwk = cohen_kappa_score(all_labels, all_predictions, weights='quadratic')

print(f"Quadratic Weighted Kappa: {qwk}")


Quadratic Weighted Kappa: 0.23468232034551595


#### **Se guardan los resultados**

En esta sección se prepara un dataset con las predicciones de cada observación del testing set que cuenta con una descripción y se la guarda en un archivo .csv para luego combinarla con las predicciones de los otros modelos. También es posible guardar el modelo entrenado en caso de querer utilizarlo para futuras observaciones.

In [ ]:
# Se crea un data frame con los resultados del texto y se guarda este dataset
pred_txt = pd.DataFrame({
    "PetID" : test_df["PetID"],
    "txt_Pred" : all_predictions,
    "txt_Clase_0" : [p[0] for p in all_probabilities],
    "txt_Clase_1" : [p[1] for p in all_probabilities],
    "txt_Clase_2" : [p[2] for p in all_probabilities],
    "txt_Clase_3" : [p[3] for p in all_probabilities],
    "txt_Clase_4" : [p[4] for p in all_probabilities]
})

pred_txt.to_csv("pred_txt_base.csv", index=False)

In [ ]:
# Se guardan los pesos obtenidos con el fine tuning
torch.save(model.state_dict(), "modelo_traduccion_20epoch_1semilla.pth")